In [ ]:
"""
3CE 问卷：认知 -> 首购 -> 经常购买漏斗。

核心人群严格定义为：
    “三线及以下/县城” AND “听说过 3CE 但从未购买”。

程序先锁定这一认知—购买断点人群，再验证其是否：
    偏好韩系文化、缺少线下试妆触点、希望真实/参与式体验，
并继续比较年龄、内容平台、购买动机、购买困难、品牌期待等行为特点。

运行：
    python 3ce_survey_funnel_analysis.py survey.xlsx --sheet 0 --out analysis_output
    python 3ce_survey_funnel_analysis.py survey.csv --out analysis_output

依赖：
    pip install pandas numpy matplotlib openpyxl

说明：
1. Q2 当前把“三线及以下城市/县城”合并在一个选项里，所以结果不能继续拆成三线、四线。
2. “线下触点缺口”用 Q3 未选择“线下试妆”作为代理，不等同于受访者绝对没有任何线下接触。
3. “真实参与体验”用试用、真实用户妆效、城市/校园活动、共创、投票等选择作为代理。
4. 程序不会硬编码“核心人群偏好韩系”的结论，而是用回收数据验证并量化。
"""

In [ ]:
from __future__ import annotations

In [ ]:
import argparse
import math
import re
from pathlib import Path
from typing import Iterable

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:  # 图表是可选输出；没有 matplotlib 时仍完成所有表格和摘要分析。
    plt = None

In [ ]:
# -----------------------------------------------------------------------------
# 1) 配置：左侧是程序内部字段，右侧关键词用于自动匹配问卷导出的中文列名。
#    如果自动匹配失败，请直接把关键词改成导出表中的完整列名。
# -----------------------------------------------------------------------------
QUESTION_KEYWORDS: dict[str, list[str]] = {
    "age": ["您的年龄"],
    "city": ["您所在城市"],
    "platforms": ["平时接触美妆内容的平台"],
    "buy_reason": ["购买彩妆产品最主要的原因"],
    "difficulty": ["购买彩妆时最大的困难"],
    "recommendation_mismatch": ["看到别人推荐", "购买后发现不适合自己"],
    "brand_job": ["美妆品牌只能解决一个问题"],
    "awareness": ["对3CE的了解程度"],
    "q10a_repeat_reason": ["持续购买3CE的主要原因"],
    "q11a_more_frequent": ["更频繁地购买3CE"],
    "q10b_lapsed_reason": ["没有继续购买或很少购买3CE"],
    "q11b_reactivate": ["重新关注或购买3CE"],
    "q10c_never_bought_reason": ["知道3CE但一直没有购买"],
    "q11c_first_purchase": ["首次购买3CE的可能性"],
    "q10d_new_brand": ["开始关注一个新的彩妆品牌"],
    "usual_brands": ["通常使用的彩妆品牌"],
    "k_content": ["是否喜欢韩剧"],
    "k_elements": ["韩剧中的哪些元素"],
    "role_aspiration": ["设想过成为", "喜欢的角色"],
    "personal_info": ["帮助您找到自己的风格", "了解哪些信息"],
    "q16a_k_role": ["韩剧角色测试"],
    "q16b_scene": ["场景妆容助手"],
    "q16c_ai_style": ["AI个人风格探索"],
    "q16d_community": ["女性圈层社区"],
    "q16e_k_trend": ["韩系潮流实验室"],
    "makeup_value": ["化妆最大的价值"],
    "long_term_need": ["美妆品牌长期陪伴", "希望它提供什么"],
}

In [ ]:
# 每个内部字段可以有多组“同时包含”的关键词；任意一组匹配即可。
# 这样既兼容完整题干，也兼容“Q2｜城市层级”一类简写表头。
QUESTION_ALIASES: dict[str, list[list[str]]] = {
    "age": [["您的年龄"], ["Q1", "年龄"], ["年龄"]],
    "city": [["您所在城市"], ["Q2", "城市"], ["城市层级"], ["城市级别"], ["城市等级"]],
    "platforms": [["平时接触美妆内容的平台"], ["Q3", "平台"], ["美妆内容", "平台"]],
    "buy_reason": [["购买彩妆产品最主要的原因"], ["Q4", "原因"], ["彩妆购买", "动机"]],
    "difficulty": [["购买彩妆时最大的困难"], ["Q6", "困难"], ["购买", "困难"]],
    "recommendation_mismatch": [
        ["看到别人推荐", "购买后发现不适合"],
        ["Q7", "不适合"],
        ["种草", "不适合"],
    ],
    "brand_job": [
        ["美妆品牌只能解决一个问题"],
        ["Q8", "品牌"],
        ["品牌", "解决的问题"],
        ["最希望品牌", "帮助"],
    ],
    "awareness": [
        ["对3CE的了解程度"],
        ["Q9", "3CE"],
        ["3CE", "认知程度"],
        ["3CE", "了解程度"],
    ],
    "q10a_repeat_reason": [
        ["持续购买3CE的主要原因"],
        ["Q10A", "原因"],
        ["持续购买", "原因"],
    ],
    "q11a_more_frequent": [
        ["更频繁地购买3CE"],
        ["Q11A"],
        ["促使", "频繁购买"],
    ],
    "q10b_lapsed_reason": [
        ["没有继续购买或很少购买3CE"],
        ["Q10B", "原因"],
        ["很少购买", "原因"],
    ],
    "q11b_reactivate": [
        ["重新关注或购买3CE"],
        ["Q11B"],
        ["重新关注", "方式"],
    ],
    "q10c_never_bought_reason": [
        ["知道3CE但一直没有购买"],
        ["Q10C", "原因"],
        ["未购买3CE", "原因"],
    ],
    "q11c_first_purchase": [
        ["首次购买3CE的可能性"],
        ["Q11C"],
        ["首次购买", "体验"],
    ],
    # 不用 Q10D 单独匹配，因为部分问卷把“常用品牌”也误标为 Q10D。
    "q10d_new_brand": [
        ["开始关注一个新的彩妆品牌"],
        ["新的彩妆品牌", "关注"],
        ["新彩妆品牌", "关注因素"],
    ],
    "usual_brands": [["通常使用的彩妆品牌"], ["常用", "彩妆品牌"]],
    "k_content": [
        ["是否喜欢韩剧"],
        ["Q12", "韩"],
        ["韩剧", "观看频率"],
        ["韩系影视", "观看"],
    ],
    "k_elements": [["韩剧中的哪些元素"], ["Q13"], ["影视", "兴趣元素"]],
    "role_aspiration": [["设想过成为", "喜欢的角色"], ["Q14"], ["角色", "想象"]],
    "personal_info": [
        ["帮助您找到自己的风格", "了解哪些信息"],
        ["Q15"],
        ["品牌了解的信息"],
    ],
    "q16a_k_role": [["韩剧角色测试"], ["Q16A"]],
    "q16b_scene": [["场景妆容助手"], ["Q16B"]],
    "q16c_ai_style": [["AI个人风格探索"], ["Q16C"]],
    "q16d_community": [["女性圈层社区"], ["Q16D"]],
    "q16e_k_trend": [["韩系潮流实验室"], ["Q16E"]],
    "makeup_value": [["化妆最大的价值"], ["Q17"], ["化妆", "价值"]],
    "long_term_need": [
        ["美妆品牌长期陪伴", "希望它提供什么"],
        ["Q18"],
        ["长期陪伴", "提供"],
    ],
}

In [ ]:
MULTI_FIELDS = [
    "platforms",
    "buy_reason",
    "difficulty",
    "q10a_repeat_reason",
    "q11a_more_frequent",
    "q10b_lapsed_reason",
    "q11b_reactivate",
    "q10c_never_bought_reason",
    "q11c_first_purchase",
    "q10d_new_brand",
    "k_elements",
    "personal_info",
]

In [ ]:
Q16_FIELDS = [
    "q16a_k_role",
    "q16b_scene",
    "q16c_ai_style",
    "q16d_community",
    "q16e_k_trend",
]

In [ ]:
Q16_LABELS = {
    "q16a_k_role": "韩剧角色测试",
    "q16b_scene": "场景妆容助手",
    "q16c_ai_style": "AI个人风格探索",
    "q16d_community": "3CE女性圈层社区",
    "q16e_k_trend": "韩系潮流实验室",
}

In [ ]:
EVIDENCE_LABELS = {
    "k_content_interest": "喜欢/观看韩剧或韩系影视",
    "korean_concept_high_interest": "韩剧角色测试或韩系潮流实验室高兴趣(4-5分)",
    "no_reported_offline_touch": "未报告线下试妆触点",
    "real_participatory_experience_need": "希望真实/参与式体验",
    "offline_gap_and_experience_need": "无报告线下触点且希望真实参与体验",
    "personalized_fit_need": "个性化适配需求",
    "scene_solution_need": "生活场景妆容方案需求",
    "virtual_ai_experience_need": "虚拟试妆或AI体验需求",
    "community_co_creation_need": "社区/共创参与需求",
}

In [ ]:
BEHAVIOR_FIELDS = [
    "age",
    "platforms",
    "buy_reason",
    "difficulty",
    "recommendation_mismatch",
    "brand_job",
    # 核心人群的断点原因和首购驱动，应与其他城市的同类断点人群比较。
    "q10c_never_bought_reason",
    "q11c_first_purchase",
    "k_content",
    "k_elements",
    "role_aspiration",
    "personal_info",
    "makeup_value",
]

In [ ]:
BEHAVIOR_LABELS = {
    "age": "Q1 年龄",
    "platforms": "Q3 美妆内容接触平台",
    "buy_reason": "Q4 彩妆购买动机",
    "difficulty": "Q6 购买困难",
    "recommendation_mismatch": "Q7 被种草后发现不适合",
    "brand_job": "Q8 最希望品牌解决的问题",
    "q10c_never_bought_reason": "Q10C 知道3CE但未购买的原因",
    "q11c_first_purchase": "Q11C 提升首次购买可能性的体验",
    "k_content": "Q12 韩剧/韩系影视观看频率",
    "k_elements": "Q13 影视兴趣元素",
    "role_aspiration": "Q14 成为喜欢角色的想象",
    "personal_info": "Q15 希望品牌了解的信息",
    "makeup_value": "Q17 化妆的最大价值",
}

In [ ]:
MULTI_SPLIT_RE = re.compile(r"\s*(?:\||┋|;|；|,|，|、|\n|\r)+\s*")

In [ ]:
def clean_text(value: object) -> str:
    """统一中英文空格、全半角括号等，方便匹配。"""
    if pd.isna(value):
        return ""
    text = str(value).strip()
    text = text.replace("（", "(").replace("）", ")").replace("：", ":")
    return re.sub(r"\s+", "", text)

In [ ]:
def read_data(path: Path, sheet: str | int = 0) -> pd.DataFrame:
    suffix = path.suffix.lower()
    if suffix in {".xlsx", ".xls"}:
        df = pd.read_excel(path, sheet_name=sheet)
    elif suffix == ".csv":
        try:
            df = pd.read_csv(path, encoding="utf-8-sig")
        except UnicodeDecodeError:
            df = pd.read_csv(path, encoding="gb18030")
    else:
        raise ValueError("仅支持 .xlsx、.xls 或 .csv 文件")

    df = df.dropna(how="all").reset_index(drop=True)
    if df.empty:
        raise ValueError("数据文件没有有效答卷。")
    return df

In [ ]:
def find_column(columns: Iterable[object], keywords: list[str]) -> str | None:
    """返回同时包含所有关键词的首个列名。"""
    normalized = [(str(col), clean_text(col).lower()) for col in columns]
    wanted = [clean_text(k).lower() for k in keywords]
    matches = [original for original, norm in normalized if all(k in norm for k in wanted)]
    if len(matches) > 1:
        print(f"[提醒] 关键词 {keywords} 匹配到多列，默认使用第一列：{matches[0]}")
    return matches[0] if matches else None

In [ ]:
def find_column_aliases(columns: Iterable[object], alias_groups: list[list[str]]) -> str | None:
    """多组别名中任意一组匹配即可；优先采用更完整、更靠前的匹配。"""
    normalized = [(str(col), clean_text(col).lower()) for col in columns]
    candidates: dict[str, float] = {}
    alias_count = len(alias_groups)
    for alias_index, keywords in enumerate(alias_groups):
        wanted = [clean_text(k).lower() for k in keywords]
        for original, norm in normalized:
            if all(keyword in norm for keyword in wanted):
                # 完整关键词长度体现匹配特异性；靠前的别名组获得轻微优先级。
                score = sum(len(keyword) for keyword in wanted) + (alias_count - alias_index) / 100
                candidates[original] = max(candidates.get(original, -1), score)
    if not candidates:
        return None
    ranked = sorted(candidates.items(), key=lambda item: (-item[1], list(map(str, columns)).index(item[0])))
    if len(ranked) > 1 and ranked[0][1] == ranked[1][1]:
        print(f"[提醒] 多个表头同等匹配，默认使用：{ranked[0][0]}")
    return ranked[0][0]

In [ ]:
def build_column_map(df: pd.DataFrame) -> dict[str, str | None]:
    mapping = {
        field: find_column_aliases(df.columns, QUESTION_ALIASES.get(field, [keywords]))
        for field, keywords in QUESTION_KEYWORDS.items()
    }
    print("\n=== 列名识别结果 ===")
    for field, column in mapping.items():
        print(f"{field:28s} -> {column or '[未找到]'}")
    return mapping

In [ ]:
def text_series(df: pd.DataFrame, column: str | None) -> pd.Series:
    if column is None:
        return pd.Series(pd.NA, index=df.index, dtype="string")
    out = df[column].astype("string").str.strip()
    return out.mask(out.eq(""), pd.NA)

In [ ]:
def contains_any(series: pd.Series, keywords: Iterable[str]) -> pd.Series:
    """可空布尔值：原值为空时返回 NA，而不是 False。"""
    pattern = "|".join(re.escape(k) for k in keywords)
    result = series.str.contains(pattern, case=False, regex=True, na=False).astype("boolean")
    return result.mask(series.isna(), pd.NA)

In [ ]:
def combine_or(*signals: pd.Series) -> pd.Series:
    """多个可空布尔条件取 OR；若全部缺失，则结果仍为 NA。"""
    frame = pd.concat(signals, axis=1)
    result = frame.fillna(False).astype(bool).any(axis=1).astype("boolean")
    return result.mask(frame.notna().sum(axis=1).eq(0), pd.NA)

In [ ]:
def score_series(df: pd.DataFrame, column: str | None) -> pd.Series:
    """兼容单元格为 1、'5分'、'5 ○' 等格式。"""
    raw = text_series(df, column)
    extracted = raw.str.extract(r"(?<!\d)([1-5])(?!\d)", expand=False)
    direct = pd.to_numeric(raw, errors="coerce")
    return direct.fillna(pd.to_numeric(extracted, errors="coerce"))

In [ ]:
def split_choices(value: object) -> list[str]:
    if pd.isna(value) or not str(value).strip():
        return []
    pieces = MULTI_SPLIT_RE.split(str(value).strip())
    cleaned = [p.strip(" □○\t") for p in pieces if p.strip(" □○\t")]
    # 开放填写的“其他”内容不直接暴露，也避免每条自由文本都被当成独立选项。
    return ["其他" if p.startswith("其他") else p for p in cleaned]

In [ ]:
def safe_rate(numerator: int | float, denominator: int | float) -> float:
    return float(numerator / denominator) if denominator else np.nan

In [ ]:
def two_proportion_test(success_a: int, n_a: int, success_b: int, n_b: int) -> tuple[float, float]:
    """返回 A-B 的比例差和双侧 p 值；不依赖 scipy。"""
    if n_a == 0 or n_b == 0:
        return np.nan, np.nan
    p_a, p_b = success_a / n_a, success_b / n_b
    pooled = (success_a + success_b) / (n_a + n_b)
    se = math.sqrt(max(pooled * (1 - pooled) * (1 / n_a + 1 / n_b), 0))
    if se == 0:
        return p_a - p_b, np.nan
    z = (p_a - p_b) / se
    p_value = math.erfc(abs(z) / math.sqrt(2))
    return p_a - p_b, p_value

In [ ]:
def top_choices(df: pd.DataFrame, column: str | None, mask: pd.Series, question: str) -> pd.DataFrame:
    if column is None:
        return pd.DataFrame(columns=["题目", "选项", "选择人数", "有效答题人数", "选择率"])
    valid = mask & df[column].notna() & df[column].astype(str).str.strip().ne("")
    base = int(valid.sum())
    counts: dict[str, int] = {}
    for value in df.loc[valid, column]:
        for choice in set(split_choices(value)):
            counts[choice] = counts.get(choice, 0) + 1
    rows = [
        {
            "题目": question,
            "选项": choice,
            "选择人数": count,
            "有效答题人数": base,
            "选择率": safe_rate(count, base),
        }
        for choice, count in counts.items()
    ]
    return pd.DataFrame(rows).sort_values(["选择率", "选择人数"], ascending=False)

In [ ]:
def make_signals(df: pd.DataFrame, col: dict[str, str | None]) -> pd.DataFrame:
    signals = pd.DataFrame(index=df.index)

    city = text_series(df, col["city"])
    awareness = text_series(df, col["awareness"])
    platforms = text_series(df, col["platforms"])
    k_content = text_series(df, col["k_content"])
    brand_job = text_series(df, col["brand_job"])

    # Q2 的现有选项只能分析“三线及以下/县城”，无法从中再拆四线。
    signals["core_city"] = contains_any(city, ["三线及以下", "三线", "四线", "五线", "县城"])

    frequent = contains_any(awareness, ["经常购买"])
    bought_once = contains_any(awareness, ["买过1-2次", "买过1—2次", "买过1–2次"])
    heard_only = contains_any(awareness, ["听说过但没有购买", "听说过但没购买"])
    unaware = contains_any(awareness, ["完全不了解"])

    signals["frequent_buyer"] = frequent
    signals["ever_bought"] = combine_or(frequent, bought_once)
    signals["aware"] = combine_or(frequent, bought_once, heard_only)
    signals["aware_nonbuyer"] = heard_only
    signals["unaware"] = unaware
    # 本项目的核心人群不是全部低线城市用户，而是城市层级与漏斗断点的交集。
    signals["target_core"] = (
        signals["core_city"] & signals["aware_nonbuyer"]
    ).astype("boolean")
    signals.loc[
        signals[["core_city", "aware_nonbuyer"]].isna().any(axis=1),
        "target_core",
    ] = pd.NA

    # Q3 是“平时接触内容的平台”，因此这里严格命名为“未报告线下试妆触点”。
    offline_selected = contains_any(platforms, ["线下试妆"])
    signals["no_reported_offline_touch"] = (~offline_selected).mask(offline_selected.isna(), pd.NA)

    signals["k_content_interest"] = contains_any(k_content, ["经常观看", "偶尔观看"])

    q16a_high = (score_series(df, col["q16a_k_role"]) >= 4).astype("boolean")
    q16e_high = (score_series(df, col["q16e_k_trend"]) >= 4).astype("boolean")
    q16a_high = q16a_high.mask(score_series(df, col["q16a_k_role"]).isna(), pd.NA)
    q16e_high = q16e_high.mask(score_series(df, col["q16e_k_trend"]).isna(), pd.NA)
    signals["korean_concept_high_interest"] = combine_or(q16a_high, q16e_high)

    branch_columns = [
        col["q11a_more_frequent"],
        col["q11b_reactivate"],
        col["q11c_first_purchase"],
        col["q10d_new_brand"],
    ]
    branch_series = [text_series(df, c) for c in branch_columns]

    personalized_parts = [
        contains_any(brand_job, ["找到适合自己的产品"]),
        *[
            contains_any(s, ["个性化", "适合我的", "适合自己", "特征推荐", "个人风格档案", "分析适合"])
            for s in branch_series
        ],
    ]
    ai_score = score_series(df, col["q16c_ai_style"])
    ai_high = (ai_score >= 4).astype("boolean").mask(ai_score.isna(), pd.NA)
    signals["personalized_fit_need"] = combine_or(*personalized_parts, ai_high)

    scene_parts = [
        contains_any(brand_job, ["不同生活场景", "不同生活情景", "生活场景下的形象"]),
        *[contains_any(s, ["场景", "情景", "完整妆容", "妆容方案"]) for s in branch_series],
    ]
    scene_score = score_series(df, col["q16b_scene"])
    scene_high = (scene_score >= 4).astype("boolean").mask(scene_score.isna(), pd.NA)
    signals["scene_solution_need"] = combine_or(*scene_parts, scene_high)

    real_experience_keywords = [
        "试用",
        "真实用户",
        "城市限定",
        "校园限定",
        "共同设计",
        "共创新品",
        "用户共创",
        "共创",
        "投票",
        "参与感",
        "能感知产品在自己脸上的效果",
    ]
    signals["real_participatory_experience_need"] = combine_or(
        *[contains_any(s, real_experience_keywords) for s in branch_series]
    )

    signals["virtual_ai_experience_need"] = combine_or(
        *[contains_any(s, ["虚拟试妆", "AI", "人工智能"]) for s in branch_series],
        ai_high,
    )

    community_score = score_series(df, col["q16d_community"])
    community_high = (community_score >= 4).astype("boolean").mask(community_score.isna(), pd.NA)
    signals["community_co_creation_need"] = combine_or(
        *[contains_any(s, ["共同设计", "共创", "投票", "参与感", "限定活动"]) for s in branch_series],
        community_high,
    )

    signals["offline_gap_and_experience_need"] = (
        signals["no_reported_offline_touch"]
        & signals["real_participatory_experience_need"]
    ).astype("boolean")
    signals.loc[
        signals[["no_reported_offline_touch", "real_participatory_experience_need"]].isna().any(axis=1),
        "offline_gap_and_experience_need",
    ] = pd.NA
    return signals

In [ ]:
def funnel_table(signals: pd.DataFrame) -> pd.DataFrame:
    groups = {
        "全部样本": pd.Series(True, index=signals.index),
        "三线及以下/县城": signals["core_city"].fillna(False),
        "一线/新一线/二线": (~signals["core_city"]).fillna(False),
    }
    rows = []
    for name, mask in groups.items():
        valid_awareness = mask & signals["aware"].notna()
        n = int(valid_awareness.sum())
        aware = int((valid_awareness & signals["aware"].fillna(False)).sum())
        bought = int((valid_awareness & signals["ever_bought"].fillna(False)).sum())
        frequent = int((valid_awareness & signals["frequent_buyer"].fillna(False)).sum())
        aware_nonbuyer = int((valid_awareness & signals["aware_nonbuyer"].fillna(False)).sum())
        rows.append(
            {
                "人群": name,
                "有效样本数": n,
                "认知人数": aware,
                "购买人数": bought,
                "经常购买人数": frequent,
                "认知未购人数": aware_nonbuyer,
                "3CE认知率": safe_rate(aware, n),
                "购买渗透率": safe_rate(bought, n),
                "认知→购买转化率": safe_rate(bought, aware),
                "认知未购断点率": safe_rate(aware_nonbuyer, aware),
                "购买→经常购买率": safe_rate(frequent, bought),
            }
        )
    return pd.DataFrame(rows)

In [ ]:
def breakpoint_source_table(signals: pd.DataFrame) -> pd.DataFrame:
    valid_city = signals["core_city"].notna()
    breakpoint = signals["aware_nonbuyer"].fillna(False)
    core = signals["core_city"].fillna(False)

    total_n = int(valid_city.sum())
    breakpoint_n = int((valid_city & breakpoint).sum())
    core_total = int((valid_city & core).sum())
    core_breakpoint = int((valid_city & breakpoint & core).sum())

    sample_share = safe_rate(core_total, total_n)
    breakpoint_share = safe_rate(core_breakpoint, breakpoint_n)
    over_index = safe_rate(breakpoint_share, sample_share)

    return pd.DataFrame(
        [
            {
                "断点定义": "听说过3CE但从未购买",
                "断点总人数": breakpoint_n,
                "其中三线及以下/县城人数": core_breakpoint,
                "三线及以下/县城在断点中的占比": breakpoint_share,
                "三线及以下/县城在总样本中的占比": sample_share,
                "城市贡献过度集中指数": over_index,
                "指数解释": ">1 表示该城市人群在断点中相对过度集中",
            }
        ]
    )

In [ ]:
def need_profile(signals: pd.DataFrame) -> pd.DataFrame:
    label_map = {
        "k_content_interest": "喜欢/观看韩剧或韩系影视",
        "korean_concept_high_interest": "韩剧角色测试或韩系潮流实验室高兴趣(4-5分)",
        "personalized_fit_need": "个性化适配需求",
        "scene_solution_need": "生活场景妆容方案需求",
        "real_participatory_experience_need": "真实/参与式体验需求",
        "virtual_ai_experience_need": "虚拟试妆或AI体验需求",
        "community_co_creation_need": "社区/共创参与需求",
        "no_reported_offline_touch": "未报告线下试妆触点",
        "offline_gap_and_experience_need": "无报告线下触点且想要真实参与体验",
    }
    rows = []
    for field, label in label_map.items():
        for segment_name, segment in {
            "三线及以下/县城": signals["core_city"].fillna(False),
            "一线/新一线/二线": (~signals["core_city"]).fillna(False),
        }.items():
            valid = segment & signals[field].notna()
            yes = int((valid & signals[field].fillna(False)).sum())
            rows.append(
                {
                    "指标": label,
                    "人群": segment_name,
                    "符合人数": yes,
                    "有效基数": int(valid.sum()),
                    "比例": safe_rate(yes, int(valid.sum())),
                }
            )

    result = pd.DataFrame(rows)
    comparisons = []
    for label, part in result.groupby("指标", sort=False):
        a = part.loc[part["人群"].eq("三线及以下/县城")].iloc[0]
        b = part.loc[part["人群"].eq("一线/新一线/二线")].iloc[0]
        diff, p_value = two_proportion_test(
            int(a["符合人数"]), int(a["有效基数"]), int(b["符合人数"]), int(b["有效基数"])
        )
        comparisons.append({"指标": label, "核心人群-其他城市差值": diff, "双侧p值": p_value})
    return result.merge(pd.DataFrame(comparisons), on="指标", how="left")

In [ ]:
def q16_table(df: pd.DataFrame, col: dict[str, str | None], core_mask: pd.Series) -> pd.DataFrame:
    rows = []
    for field in Q16_FIELDS:
        scores = score_series(df, col[field])
        for group_name, mask in {
            "三线及以下/县城": core_mask,
            "一线/新一线/二线": ~core_mask,
        }.items():
            values = scores[mask & scores.notna()]
            rows.append(
                {
                    "体验概念": Q16_LABELS[field],
                    "人群": group_name,
                    "有效样本数": int(values.size),
                    "平均分": values.mean(),
                    "4-5分比例": (values >= 4).mean() if len(values) else np.nan,
                }
            )
    return pd.DataFrame(rows)

In [ ]:
def comparison_groups(signals: pd.DataFrame) -> dict[str, pd.Series]:
    """构造互有解释力的对照组。"""
    core_city = signals["core_city"].fillna(False)
    breakpoint = signals["aware_nonbuyer"].fillna(False)
    bought = signals["ever_bought"].fillna(False)
    target = core_city & breakpoint
    return {
        "目标：三线及以下认知未购": target,
        "同城已购": core_city & bought,
        "其他城市认知未购": (~core_city) & breakpoint,
        "其余样本": ~target,
    }

In [ ]:
def evidence_judgement(
    target_rate: float,
    target_n: int,
    comparisons: list[tuple[float, float]],
) -> str:
    """comparisons 中每项为（比例差，p值）。"""
    if pd.isna(target_rate):
        return "无有效数据"
    valid = [(d, p) for d, p in comparisons if pd.notna(d)]
    significant_positive = any(d > 0 and pd.notna(p) and p < 0.05 for d, p in valid)
    all_positive = bool(valid) and all(d > 0 for d, _ in valid)
    small_sample = target_n < 30

    if target_rate < 0.5:
        result = "目标组未过半，暂不支持‘多数人具有该特点’"
    elif significant_positive:
        result = "数据支持，且至少一个对照差异显著"
    elif all_positive:
        result = "方向支持，但差异未达显著"
    else:
        result = "目标组内部占比较高，但并非该人群独有"
    if small_sample:
        result = "小样本；" + result
    return result

In [ ]:
def target_evidence_table(signals: pd.DataFrame) -> pd.DataFrame:
    groups = comparison_groups(signals)
    rows = []
    for field, label in EVIDENCE_LABELS.items():
        group_stats: dict[str, tuple[int, int, float]] = {}
        for group_name, mask in groups.items():
            valid = mask & signals[field].notna()
            n = int(valid.sum())
            yes = int((valid & signals[field].fillna(False)).sum())
            group_stats[group_name] = (yes, n, safe_rate(yes, n))

        t_yes, t_n, t_rate = group_stats["目标：三线及以下认知未购"]
        c_yes, c_n, c_rate = group_stats["同城已购"]
        o_yes, o_n, o_rate = group_stats["其他城市认知未购"]
        diff_city_buyer, p_city_buyer = two_proportion_test(t_yes, t_n, c_yes, c_n)
        diff_other_break, p_other_break = two_proportion_test(t_yes, t_n, o_yes, o_n)

        rows.append(
            {
                "指标": label,
                "目标组符合人数": t_yes,
                "目标组有效基数": t_n,
                "目标组比例": t_rate,
                "同城已购符合人数": c_yes,
                "同城已购有效基数": c_n,
                "同城已购比例": c_rate,
                "目标-同城已购差值": diff_city_buyer,
                "目标/同城已购提升指数": safe_rate(t_rate, c_rate),
                "目标vs同城已购p值": p_city_buyer,
                "其他城市断点符合人数": o_yes,
                "其他城市断点有效基数": o_n,
                "其他城市断点比例": o_rate,
                "目标-其他城市断点差值": diff_other_break,
                "目标/其他城市断点提升指数": safe_rate(t_rate, o_rate),
                "目标vs其他城市断点p值": p_other_break,
                "证据判断": evidence_judgement(
                    t_rate,
                    t_n,
                    [(diff_city_buyer, p_city_buyer), (diff_other_break, p_other_break)],
                ),
            }
        )
    return pd.DataFrame(rows)

In [ ]:
def choice_stats(series: pd.Series, mask: pd.Series) -> tuple[int, dict[str, int]]:
    valid = mask & series.notna() & series.astype("string").str.strip().ne("")
    base = int(valid.sum())
    counts: dict[str, int] = {}
    for value in series[valid]:
        for choice in set(split_choices(value)):
            counts[choice] = counts.get(choice, 0) + 1
    return base, counts

In [ ]:
def target_behavior_table(
    df: pd.DataFrame,
    col: dict[str, str | None],
    signals: pd.DataFrame,
) -> pd.DataFrame:
    """逐题逐选项比较目标组、同城已购者、其他城市断点人群。"""
    groups = comparison_groups(signals)
    rows = []
    for order, field in enumerate(BEHAVIOR_FIELDS, start=1):
        column = col.get(field)
        if column is None:
            continue
        series = text_series(df, column)
        stats = {name: choice_stats(series, mask) for name, mask in groups.items()}
        all_options = sorted({option for _, counts in stats.values() for option in counts})
        for option in all_options:
            target_base, target_counts = stats["目标：三线及以下认知未购"]
            city_base, city_counts = stats["同城已购"]
            other_base, other_counts = stats["其他城市认知未购"]
            target_yes = target_counts.get(option, 0)
            city_yes = city_counts.get(option, 0)
            other_yes = other_counts.get(option, 0)
            target_rate = safe_rate(target_yes, target_base)
            city_rate = safe_rate(city_yes, city_base)
            other_rate = safe_rate(other_yes, other_base)
            city_diff, city_p = two_proportion_test(target_yes, target_base, city_yes, city_base)
            other_diff, other_p = two_proportion_test(target_yes, target_base, other_yes, other_base)
            rows.append(
                {
                    "题目顺序": order,
                    "字段": field,
                    "题目": BEHAVIOR_LABELS[field],
                    "选项": option,
                    "目标组选择人数": target_yes,
                    "目标组有效基数": target_base,
                    "目标组选择率": target_rate,
                    "同城已购选择人数": city_yes,
                    "同城已购有效基数": city_base,
                    "同城已购选择率": city_rate,
                    "目标-同城已购差值": city_diff,
                    "目标/同城已购提升指数": safe_rate(target_rate, city_rate),
                    "目标vs同城已购p值": city_p,
                    "其他城市断点选择人数": other_yes,
                    "其他城市断点有效基数": other_base,
                    "其他城市断点选择率": other_rate,
                    "目标-其他城市断点差值": other_diff,
                    "目标/其他城市断点提升指数": safe_rate(target_rate, other_rate),
                    "目标vs其他城市断点p值": other_p,
                }
            )
    if not rows:
        return pd.DataFrame()
    return pd.DataFrame(rows).sort_values(
        ["题目顺序", "目标组选择率", "目标组选择人数"],
        ascending=[True, False, False],
    )

In [ ]:
def target_q16_table(
    df: pd.DataFrame,
    col: dict[str, str | None],
    signals: pd.DataFrame,
) -> pd.DataFrame:
    groups = comparison_groups(signals)
    rows = []
    for field in Q16_FIELDS:
        scores = score_series(df, col[field])
        row: dict[str, object] = {"体验概念": Q16_LABELS[field]}
        high_stats: dict[str, tuple[int, int]] = {}
        for group_name, mask in groups.items():
            values = scores[mask & scores.notna()]
            short = {
                "目标：三线及以下认知未购": "目标组",
                "同城已购": "同城已购",
                "其他城市认知未购": "其他城市断点",
                "其余样本": "其余样本",
            }[group_name]
            n = int(values.size)
            high = int((values >= 4).sum())
            high_stats[group_name] = (high, n)
            row[f"{short}有效样本数"] = n
            row[f"{short}平均分"] = values.mean()
            row[f"{short}4-5分比例"] = safe_rate(high, n)
        t_high, t_n = high_stats["目标：三线及以下认知未购"]
        c_high, c_n = high_stats["同城已购"]
        o_high, o_n = high_stats["其他城市认知未购"]
        row["目标-同城已购平均分差"] = row["目标组平均分"] - row["同城已购平均分"]
        row["目标-其他城市断点平均分差"] = row["目标组平均分"] - row["其他城市断点平均分"]
        row["目标vs同城已购高分率p值"] = two_proportion_test(t_high, t_n, c_high, c_n)[1]
        row["目标vs其他城市断点高分率p值"] = two_proportion_test(t_high, t_n, o_high, o_n)[1]
        rows.append(row)
    return pd.DataFrame(rows).sort_values("目标组平均分", ascending=False)

In [ ]:
def plot_target_evidence(evidence: pd.DataFrame, output: Path) -> None:
    if plt is None:
        return
    plot_df = evidence.iloc[::-1]
    y = np.arange(len(plot_df))
    height = 0.24
    fig, ax = plt.subplots(figsize=(12, 7.5))
    series = [
        ("目标组比例", "目标：三线及以下认知未购", "#E65A73", -height),
        ("同城已购比例", "同城已购", "#5665D2", 0),
        ("其他城市断点比例", "其他城市认知未购", "#56A68B", height),
    ]
    for column, label, color, offset in series:
        bars = ax.barh(y + offset, plot_df[column], height, label=label, color=color)
        ax.bar_label(
            bars,
            labels=[f"{v:.0%}" if pd.notna(v) else "NA" for v in plot_df[column]],
            padding=2,
            fontsize=8,
        )
    ax.set_yticks(y, plot_df["指标"])
    ax.set_xlim(0, 1.08)
    ax.set_xlabel("符合比例")
    ax.set_title("三线及以下认知未购人群：核心假设证据与对照")
    ax.grid(axis="x", alpha=0.2)
    ax.legend(frameon=False, loc="lower right")
    fig.tight_layout()
    fig.savefig(output, dpi=180, bbox_inches="tight")
    plt.close(fig)

In [ ]:
def plot_funnel(funnel: pd.DataFrame, output: Path) -> None:
    if plt is None:
        return
    plot_df = funnel[funnel["人群"].ne("全部样本")].copy()
    stages = ["3CE认知率", "购买渗透率", "购买→经常购买率"]
    x = np.arange(len(stages))
    width = 0.36
    fig, ax = plt.subplots(figsize=(10, 5.5))
    colors = ["#E65A73", "#5665D2"]
    for i, (_, row) in enumerate(plot_df.iterrows()):
        values = [row[s] for s in stages]
        bars = ax.bar(x + (i - 0.5) * width, values, width, label=row["人群"], color=colors[i])
        ax.bar_label(bars, labels=[f"{v:.1%}" if pd.notna(v) else "NA" for v in values], padding=3)
    ax.set_xticks(x, ["认知率", "购买渗透率", "购买者中的经常购买率"])
    ax.set_ylim(0, 1.08)
    ax.set_ylabel("比例")
    ax.set_title("3CE 认知—购买漏斗：城市层级对比")
    ax.legend(frameon=False)
    ax.grid(axis="y", alpha=0.2)
    fig.tight_layout()
    fig.savefig(output, dpi=180, bbox_inches="tight")
    plt.close(fig)

In [ ]:
def plot_need_profile(profile: pd.DataFrame, output: Path) -> None:
    if plt is None:
        return
    pivot = profile.pivot(index="指标", columns="人群", values="比例")
    pivot = pivot.sort_values("三线及以下/县城", ascending=True)
    ax = pivot.plot.barh(figsize=(11, 7), color=["#5665D2", "#E65A73"])
    ax.set_xlim(0, 1.05)
    ax.set_xlabel("符合比例")
    ax.set_ylabel("")
    ax.set_title("核心人群偏好、需求与触点缺口")
    ax.grid(axis="x", alpha=0.2)
    ax.legend(frameon=False, loc="lower right")
    for container in ax.containers:
        ax.bar_label(
            container,
            labels=[f"{v:.0%}" if pd.notna(v) else "NA" for v in container.datavalues],
            padding=2,
            fontsize=8,
        )
    ax.figure.tight_layout()
    ax.figure.savefig(output, dpi=180, bbox_inches="tight")
    plt.close(ax.figure)

In [ ]:
def pct(value: float) -> str:
    return "NA" if pd.isna(value) else f"{value:.1%}"

In [ ]:
def make_summary(
    funnel: pd.DataFrame,
    breakpoint: pd.DataFrame,
    profile: pd.DataFrame,
    q16: pd.DataFrame,
) -> str:
    core_funnel = funnel.loc[funnel["人群"].eq("三线及以下/县城")].iloc[0]
    other_funnel = funnel.loc[funnel["人群"].eq("一线/新一线/二线")].iloc[0]
    bp = breakpoint.iloc[0]

    def core_metric(label: str) -> pd.Series:
        return profile[(profile["指标"].eq(label)) & (profile["人群"].eq("三线及以下/县城"))].iloc[0]

    k_drama = core_metric("喜欢/观看韩剧或韩系影视")
    k_concept = core_metric("韩剧角色测试或韩系潮流实验室高兴趣(4-5分)")
    no_offline = core_metric("未报告线下试妆触点")
    real_exp = core_metric("真实/参与式体验需求")
    overlap = core_metric("无报告线下触点且想要真实参与体验")

    q16_core = q16[q16["人群"].eq("三线及以下/县城")].sort_values("平均分", ascending=False)
    best_concept = q16_core.iloc[0] if not q16_core.empty else None

    lines = [
        "# 3CE 认知到购买断点分析",
        "",
        "## 1. 核心断点",
        "",
        f"- 三线及以下/县城人群的 3CE 认知率为 {pct(core_funnel['3CE认知率'])}，认知→购买转化率为 {pct(core_funnel['认知→购买转化率'])}，认知未购断点率为 {pct(core_funnel['认知未购断点率'])}。",
        f"- 一线/新一线/二线的认知→购买转化率为 {pct(other_funnel['认知→购买转化率'])}，可与核心人群直接比较。",
        f"- 在所有‘听说过但未购买’者中，三线及以下/县城贡献 {pct(bp['三线及以下/县城在断点中的占比'])}；其在总样本中占 {pct(bp['三线及以下/县城在总样本中的占比'])}，过度集中指数为 {bp['城市贡献过度集中指数']:.2f}。指数大于 1 才能说明其在断点中相对集中，而不只是样本量更大。",
        "",
        "## 2. 核心人群偏好和需求",
        "",
        f"- 韩系文化：{pct(k_drama['比例'])} 喜欢/观看韩剧或韩系影视，{pct(k_concept['比例'])} 对韩剧角色测试或韩系潮流实验室给出 4–5 分。",
        f"- 线下触点：{pct(no_offline['比例'])} 未在 Q3 报告线下试妆触点。该指标是问卷代理变量，不代表绝对没有线下接触。",
        f"- 真实参与：{pct(real_exp['比例'])} 选择了试用、真实用户妆效、城市/校园活动或共创投票等体验；{pct(overlap['比例'])} 同时存在‘未报告线下触点 + 想要真实参与体验’。",
    ]
    if best_concept is not None and pd.notna(best_concept["平均分"]):
        lines.append(
            f"- Q16 中核心人群评分最高的概念是“{best_concept['体验概念']}”，平均 {best_concept['平均分']:.2f} 分，4–5 分占比 {pct(best_concept['4-5分比例'])}。"
        )
    lines += [
        "",
        "## 3. 策略含义",
        "",
        "1. 若三线及以下/县城的认知未购断点率更高，应优先解决‘知道品牌但无法判断是否适合自己’，而不是只加大曝光。",
        "2. 用韩系潮流内容做兴趣入口，但把内容落到个人色号、气质和生活场景方案，推动从观看到决策。",
        "3. 针对线下触点缺口，设计可验证、可参与的体验，例如城市快闪/校园试妆、低门槛试用装、真实用户同肤色妆效、用户共创投票。",
        "4. 虚拟试妆和 AI 可承担购买前筛选，但不能替代真实试用；建议形成‘线上诊断—领取试用—反馈共创—首购转化’闭环。",
        "",
        "## 口径提醒",
        "",
        "- 当前 Q2 选项合并了‘三线及以下/县城’，无法独立证明四线城市结论。若必须拆分三线和四线，应在下一版问卷中分别设项，或收集城市名后外接城市等级表。",
        "- 多选题比例的分母是该题在对应人群中的有效答题人数。",
        "- p 值只用于辅助判断城市差异是否可能来自抽样波动；便利抽样不能据此推断总体因果。",
    ]
    return "\n".join(lines)

In [ ]:
def make_target_summary(
    signals: pd.DataFrame,
    funnel: pd.DataFrame,
    breakpoint: pd.DataFrame,
    evidence: pd.DataFrame,
    behaviors: pd.DataFrame,
    q16: pd.DataFrame,
) -> str:
    target_n = int(signals["target_core"].fillna(False).sum())
    bp = breakpoint.iloc[0]
    core_funnel = funnel.loc[funnel["人群"].eq("三线及以下/县城")].iloc[0]

    def evidence_line(label: str) -> str:
        row = evidence.loc[evidence["指标"].eq(label)].iloc[0]
        return (
            f"- **{label}**：目标组 {pct(row['目标组比例'])}（n={int(row['目标组有效基数'])}）；"
            f"同城已购者 {pct(row['同城已购比例'])}，"
            f"其他城市认知未购者 {pct(row['其他城市断点比例'])}。"
            f"判断：{row['证据判断']}。"
        )

    primary_labels = [
        "喜欢/观看韩剧或韩系影视",
        "韩剧角色测试或韩系潮流实验室高兴趣(4-5分)",
        "未报告线下试妆触点",
        "希望真实/参与式体验",
        "无报告线下触点且希望真实参与体验",
    ]

    behavior_lines: list[str] = []
    if not behaviors.empty:
        candidates = behaviors.copy()
        # 非核心假设题；优先使用同城已购作为对照，若无基数则用其他城市断点。
        candidates = candidates[~candidates["字段"].isin(["k_content"])]
        candidates["可用对照差值"] = np.where(
            candidates["同城已购有效基数"].ge(10),
            candidates["目标-同城已购差值"],
            candidates["目标-其他城市断点差值"],
        )
        candidates["可用对照名称"] = np.where(
            candidates["同城已购有效基数"].ge(10), "同城已购", "其他城市断点"
        )
        candidates = candidates[
            candidates["目标组有效基数"].ge(10)
            & candidates["目标组选择率"].ge(0.15)
            & candidates["可用对照差值"].notna()
        ]
        # 每道题最多保留一个最突出的特点，避免同一道多选题占满摘要。
        candidates = (
            candidates.sort_values(["可用对照差值", "目标组选择率"], ascending=False)
            .groupby("题目", as_index=False, sort=False)
            .head(1)
            .head(6)
        )
        for _, row in candidates.iterrows():
            behavior_lines.append(
                f"- {row['题目']}：**{row['选项']}**，目标组选择率 {pct(row['目标组选择率'])}，"
                f"较{row['可用对照名称']}高 {row['可用对照差值']:.1%}。"
            )
    if not behavior_lines:
        behavior_lines = ["- 有效样本或对照基数不足，暂不自动提炼其他差异特点；请查看完整行为特征表。"]

    q16_lines: list[str] = []
    for _, row in q16.head(3).iterrows():
        if pd.notna(row["目标组平均分"]):
            q16_lines.append(
                f"- {row['体验概念']}：目标组平均 {row['目标组平均分']:.2f} 分，"
                f"4–5分占比 {pct(row['目标组4-5分比例'])}。"
            )
    if not q16_lines:
        q16_lines = ["- Q16 暂无有效评分。"]

    lines = [
        "# 3CE 三线及以下认知—购买断点人群分析",
        "",
        "## 1. 人群定义与断点规模",
        "",
        "核心人群严格定义为：**三线及以下/县城，并且听说过3CE但从未购买**。",
        "",
        f"- 核心人群共 {target_n} 人。",
        f"- 三线及以下/县城内部的认知未购断点率为 {pct(core_funnel['认知未购断点率'])}。",
        f"- 在全部认知未购者中，该城市人群占 {pct(bp['三线及以下/县城在断点中的占比'])}；相对其总样本占比的过度集中指数为 {bp['城市贡献过度集中指数']:.2f}。",
        "",
        "## 2. 三个核心假设的证据",
        "",
        "这里的‘证明’分成两层：目标组内部是否占多数，以及相对同城已购者/其他城市断点人群是否更突出。",
        "",
        *[evidence_line(label) for label in primary_labels],
        "",
        "## 3. 其他行为特点",
        "",
        *behavior_lines,
        "",
        "## 4. Q16体验概念偏好",
        "",
        *q16_lines,
        "",
        "## 5. 解读原则",
        "",
        "- ‘数据支持且差异显著’：目标组比例至少过半，并且至少一个对照差异的双侧 p<0.05。",
        "- ‘方向支持’：目标组比例过半、对照方向一致，但样本量或差异不足以达到显著。不能写成已经被严格证明。",
        "- ‘目标组内部高，但并非独有’：这个需求真实存在，但不适合作为该细分人群的独占标签。",
        "- Q3未选择‘线下试妆’只是线下触点缺口的代理指标，不代表受访者绝对没有任何线下接触。",
        "- 当前Q2只能识别‘三线及以下/县城’，不能把三线和四线分别估计。",
    ]
    return "\n".join(lines)

In [ ]:
def save_csv(df: pd.DataFrame, path: Path) -> None:
    df.to_csv(path, index=False, encoding="utf-8-sig", float_format="%.6f")

In [ ]:
def run(input_path: Path, output_dir: Path, sheet: str | int = 0) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)
    df = read_data(input_path, sheet=sheet)
    col = build_column_map(df)

    required = ["city", "awareness"]
    missing = [field for field in required if col[field] is None]
    if missing:
        actual_columns = "\n".join(f"{i + 1}. {column}" for i, column in enumerate(df.columns))
        diagnostic_path = output_dir / "列名识别诊断.txt"
        diagnostic_path.write_text(
            "未识别的核心字段：" + ", ".join(missing) + "\n\n实际表头：\n" + actual_columns,
            encoding="utf-8",
        )
        print("\n=== 数据文件中的全部实际列名 ===")
        print(actual_columns)
        raise KeyError(
            f"缺少核心字段 {missing}。完整表头已保存到：{diagnostic_path.resolve()}"
        )

    signals = make_signals(df, col)
    funnel = funnel_table(signals)
    breakpoint = breakpoint_source_table(signals)
    evidence = target_evidence_table(signals)
    behaviors = target_behavior_table(df, col, signals)
    q16 = target_q16_table(df, col, signals)

    # 只导出匿名分析标记，不复制开放题原文或其他可能识别个人的信息。
    respondent_flags = signals.copy()
    respondent_flags.insert(0, "匿名行号", np.arange(1, len(respondent_flags) + 1))

    save_csv(funnel, output_dir / "01_认知购买漏斗.csv")
    save_csv(breakpoint, output_dir / "02_断点城市来源.csv")
    save_csv(evidence, output_dir / "03_核心假设证据检验.csv")
    save_csv(behaviors, output_dir / "04_目标人群其他行为特征.csv")
    save_csv(q16, output_dir / "05_目标人群Q16体验概念评分.csv")
    save_csv(respondent_flags, output_dir / "06_匿名分析标记.csv")

    # 中文字体按 Windows/常见跨平台环境依次回退。
    if plt is not None:
        plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Noto Sans CJK SC", "Arial Unicode MS"]
        plt.rcParams["axes.unicode_minus"] = False
        plot_funnel(funnel, output_dir / "07_城市漏斗对比.png")
        plot_target_evidence(evidence, output_dir / "08_核心假设证据对比.png")
    else:
        print("[提醒] 未安装 matplotlib，已跳过 PNG 图表；CSV 和分析摘要不受影响。")

    summary = make_target_summary(signals, funnel, breakpoint, evidence, behaviors, q16)
    (output_dir / "09_分析摘要.md").write_text(summary, encoding="utf-8")

    print(f"\n分析完成：{output_dir.resolve()}")
    print(summary)

In [ ]:
def parse_sheet(value: str) -> str | int:
    return int(value) if value.isdigit() else value

In [ ]:
def choose_input_file() -> Path:
    """无命令行参数时弹出文件选择框，方便在 PyCharm 中直接点击运行。"""
    selected = ""
    try:
        import tkinter as tk
        from tkinter import filedialog

        root = tk.Tk()
        root.withdraw()
        root.attributes("-topmost", True)
        selected = filedialog.askopenfilename(
            title="请选择问卷导出的 Excel 或 CSV 文件",
            filetypes=[
                ("问卷数据", "*.xlsx *.xls *.csv"),
                ("Excel", "*.xlsx *.xls"),
                ("CSV", "*.csv"),
                ("所有文件", "*.*"),
            ],
        )
        root.destroy()
    except Exception:
        # 某些精简 Python 环境没有 tkinter，退回控制台输入。
        selected = input("请输入问卷数据文件的完整路径：").strip().strip('"')

    if not selected:
        raise SystemExit("未选择问卷数据文件，分析已取消。")
    path = Path(selected)
    if not path.is_file():
        raise SystemExit(f"找不到数据文件：{path}")
    return path

In [ ]:
def main() -> None:
    parser = argparse.ArgumentParser(description="3CE 问卷认知—购买断点分析")
    parser.add_argument(
        "input",
        nargs="?",
        type=Path,
        help="问卷导出的 .xlsx/.xls/.csv 文件；不填写时会弹出文件选择窗口",
    )
    parser.add_argument("--sheet", default="0", help="Excel 工作表序号或名称，默认 0")
    parser.add_argument("--out", type=Path, default=None, help="输出文件夹；默认保存在数据文件旁边")
    args = parser.parse_args()
    input_path = args.input if args.input is not None else choose_input_file()
    if not input_path.is_file():
        parser.error(f"找不到数据文件：{input_path}")
    output_dir = args.out or input_path.parent / f"{input_path.stem}_分析结果"
    run(input_path, output_dir, sheet=parse_sheet(args.sheet))

In [ ]:
if __name__ == "__main__":
    main()